In [1]:
!pip install -q transformers scikit-learn pandas numpy tqdm torch-ema
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.9 MB/s eta 0:00:00:00:0100:01
Mon Sep 15 17:11:28 2025       
+-------------------------------------------------------------------

In [2]:
import threading, time, sys
from datetime import datetime

def _keepalive(interval=600):  # default 10 menit
    while True:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        sys.stdout.write(f"\r💤 keepalive {ts}")
        sys.stdout.flush()
        time.sleep(interval)

t = threading.Thread(target=_keepalive, args=(600,), daemon=True)
t.start()

💤 keepalive 2025-09-15 17:11:28

In [ ]:
# !mkdir -p /kaggle/working/models
# !mkdir -p /kaggle/working/reports
# !cp /kaggle/input/go-subchannel/models/checkpoint.pt /kaggle/working/models/checkpoint.pt
# !cp /kaggle/input/go-subchannel/reports/evaluation_report_09092025170832.txt /kaggle/working/reports/evaluation_report_09092025170832.txt
# !cp /kaggle/input/go-subchannel/models/model.pt /kaggle/working/models/model.pt

In [ ]:
# import os
# os.environ["RESUME"] = "1"

In [3]:
%%writefile config.py
from os import environ as env

class Config:
    # ===== Data =====
    DATASET_DIR = "/kaggle/input/subchannel"
    TRAIN_SET_PATH = VALIDATION_SET_PATH = TEST_SET_PATH = DATASET_DIR
    TRAIN_SET_FILENAME = "train.csv"
    VALIDATION_SET_FILENAME = "val.csv"
    TEST_SET_FILENAME = "test.csv"

    # ===== Output =====
    MODEL_PATH = "/kaggle/working/models"
    EVALUATION_REPORT_PATH = "/kaggle/working/reports"
    CONFUSION_MATRIX_PATH = "/kaggle/working/reports"

    # ===== Training config =====
    SEED = int(env.get("SEED", "42"))
    EPOCH = int(env.get("EPOCH", "10"))
    MAX_LEN = int(env.get("MAX_LEN", "512"))
    BATCH_SIZE = int(env.get("BATCH_SIZE", "64"))
    LR = float(env.get("LR", "2e-5"))
    WEIGHT_DECAY = float(env.get("WEIGHT_DECAY", "0.01"))
    WARMUP_RATIO = float(env.get("WARMUP_RATIO", "0.1"))

    # ===== Hugging Face model ID =====
    HF_MODEL_ID = env.get("HF_MODEL_ID", "bert-base-multilingual-uncased")

# instantiate
config = Config()

Writing config.py


In [4]:
# %%writefile train_bert_amp.py
# -*- coding: utf-8 -*-
import os
import math
import json
import torch
import random
import joblib
import shutil
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from config import config  # uses your config.py

# sklearn
from sklearn import preprocessing
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
)

# transformers (modern)
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,   # default
)

from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.cuda.amp import autocast, GradScaler

# =========================
# Setup (Kaggle-friendly)
# =========================
os.makedirs(config.MODEL_PATH, exist_ok=True)
os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
os.makedirs(config.CONFUSION_MATRIX_PATH, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

SEED = int(config.SEED)
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = (device.type == "cuda")
torch.set_num_threads(4)

DATE = datetime.now()
VERSION = DATE.strftime("%Y%m%d%H%M%S")  # dipakai sebagai suffix artefak

# Tiny logger to file + stdout
report_path = f"{config.EVALUATION_REPORT_PATH}/evaluation_report_{VERSION}.txt"
evaluation_report = open(report_path, "w", encoding="utf-8")
def log_report(msg: str):
    print(msg)
    evaluation_report.write(msg + "\n")
    evaluation_report.flush()

evaluation_report.write(f"Trained at {DATE.strftime('%d/%m/%Y %H:%M:%S')}\n")
evaluation_report.write(f"SEED : {SEED}\n")

# =========================
# Hyperparams & model id
# =========================
MAX_LEN      = int(config.MAX_LEN)
BATCH_SIZE   = int(config.BATCH_SIZE)  # total batch; DP akan bagi otomatis per-GPU
LR           = float(config.LR)
WEIGHT_DECAY = float(config.WEIGHT_DECAY)
WARMUP_RATIO = float(config.WARMUP_RATIO)
EPOCHS       = int(config.EPOCH)
MODEL_ID     = config.HF_MODEL_ID

# Toggle-able via ENV (tanpa ubah kode)
SCHEDULER   = os.environ.get("SCHEDULER", "cosine")      # "linear" | "cosine"
MAX_NORM    = float(os.environ.get("MAX_NORM", "1.0"))   # 0=off, default 1.0
LABEL_SMOOTH= float(os.environ.get("LABEL_SMOOTH", "0")) # 0=off, 0.05 aman
USE_EMA     = int(os.environ.get("USE_EMA", "1"))        # 1=on, 0=off
EMA_DECAY   = float(os.environ.get("EMA_DECAY", "0.999"))# 0.999-0.9999 umum

# Early stopping & checkpoint
MIN_DELTA = 1e-3
PATIENCE  = int(os.environ.get("PATIENCE", "2"))
CKPT_PATH = f"{config.MODEL_PATH}/checkpoint.pt"

# ===== artefak “production-style” =====
CONFIG_DIR    = os.path.join(config.MODEL_PATH, f"config_subchannel_{VERSION}")
TOKENIZER_DIR = os.path.join(config.MODEL_PATH, f"tokenizer_subchannel_{VERSION}")
WEIGHTS_PATH  = os.path.join(config.MODEL_PATH, f"model_subchannel_{VERSION}.pth")
LE_PATH       = os.path.join(config.MODEL_PATH, "label_encoder.pkl")

# juga simpan pointer “latest” (opsional, biar mudah dipakai downstream)
CONFIG_DIR_LATEST    = os.path.join(config.MODEL_PATH, "config_subchannel_latest")
TOKENIZER_DIR_LATEST = os.path.join(config.MODEL_PATH, "tokenizer_subchannel_latest")
WEIGHTS_PATH_LATEST  = os.path.join(config.MODEL_PATH, "model_subchannel_latest.pth")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# === header run ===
log_report("===== Training Parameters =====")
log_report(f"Model ID         : {MODEL_ID}")
log_report(f"Device / AMP     : {device} / {AMP_ENABLED}")
log_report(f"Seed             : {SEED}")
log_report(f"Max length       : {MAX_LEN}")
log_report(f"Batch size       : {BATCH_SIZE} (total)")
log_report(f"Learning rate    : {LR}")
log_report(f"Weight decay     : {WEIGHT_DECAY}")
log_report(f"Warmup ratio     : {WARMUP_RATIO}")
log_report(f"Epochs (max)     : {EPOCHS}")
log_report(f"Patience         : {PATIENCE}")
log_report(f"Min delta (loss) : {MIN_DELTA}")
log_report(f"Scheduler        : {SCHEDULER}")
log_report(f"Grad clip maxnorm: {MAX_NORM}")
log_report(f"Label smoothing  : {LABEL_SMOOTH}")
log_report(f"EMA              : {'ON' if USE_EMA else 'OFF'} (decay={EMA_DECAY})")
log_report(f"Artifacts VER    : {VERSION}")
log_report("================================")

# =========================
# Encode helper (chunked)
# =========================
def encode_texts(texts, max_len=MAX_LEN, chunk_size=10000):
    all_input_ids, all_attn = [], []
    n = len(texts)
    for i in tqdm(range(0, n, chunk_size), desc="Tokenizing", total=math.ceil(n / chunk_size)):
        batch = list(map(str, texts[i:i+chunk_size]))
        enc = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_attention_mask=True,
            return_tensors="np",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
    return np.vstack(all_input_ids), np.vstack(all_attn)

# =========================
# Load dataset
# =========================
trainset = pd.read_csv(f"{config.TRAIN_SET_PATH}/{config.TRAIN_SET_FILENAME}")
valset   = pd.read_csv(f"{config.VALIDATION_SET_PATH}/{config.VALIDATION_SET_FILENAME}")
testset  = pd.read_csv(f"{config.TEST_SET_PATH}/{config.TEST_SET_FILENAME}")

trainset = trainset.dropna().reset_index(drop=True)
valset   = valset.dropna().reset_index(drop=True)
testset  = testset.dropna().reset_index(drop=True)

# trainset = trainset.sample(n=1024, random_state=42).reset_index(drop=True)
# valset   = valset.sample(n=256, random_state=42).reset_index(drop=True)
# testset  = testset.sample(n=512, random_state=42).reset_index(drop=True)

# =========================
# Label encoding
# =========================
train_sentences = trainset["content"].values
train_labels_raw = trainset["subchannel"].values

le = preprocessing.LabelEncoder()
le.fit(train_labels_raw)
print(f"Total Subchannel: {len(le.classes_)}")
print(list(le.classes_))

# save encoder + mapping
joblib.dump(le, LE_PATH)
id2label = {i: cls for i, cls in enumerate(le.classes_)}
label2id = {v: i for i, v in id2label.items()}

train_labels = le.transform(train_labels_raw)

# =========================
# Encode splits
# =========================
print("START PROCESSING TRAIN SET . . . ")
train_input_ids, train_attention_masks = encode_texts(train_sentences, max_len=MAX_LEN)
train_inputs = torch.tensor(train_input_ids, dtype=torch.long)
train_masks  = torch.tensor(train_attention_masks, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.long)
train_data   = TensorDataset(train_inputs, train_masks, train_labels)

print("START PROCESSING VALIDATION SET . . . ")
val_sentences = valset["content"].values
val_labels = torch.tensor(le.transform(valset["subchannel"].values), dtype=torch.long)
val_input_ids, val_attention_masks = encode_texts(val_sentences, max_len=MAX_LEN)
val_inputs = torch.tensor(val_input_ids, dtype=torch.long)
val_masks  = torch.tensor(val_attention_masks, dtype=torch.long)
validation_data = TensorDataset(val_inputs, val_masks, val_labels)

print("START PROCESSING TEST SET . . . ")
test_sentences = testset["content"].values
test_labels = torch.tensor(le.transform(testset["subchannel"].values), dtype=torch.long)
test_input_ids, test_attention_masks = encode_texts(test_sentences, max_len=MAX_LEN)
test_inputs = torch.tensor(test_input_ids, dtype=torch.long)
test_masks  = torch.tensor(test_attention_masks, dtype=torch.long)
test_data   = TensorDataset(test_inputs, test_masks, test_labels)

# =========================
# Dataloaders (ngebut)
# =========================
def build_train_loader():
    train_sampler = RandomSampler(train_data)
    return DataLoader(
        train_data,
        sampler=train_sampler,
        batch_size=BATCH_SIZE,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(
    validation_data,
    sampler=validation_sampler,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(
    test_data,
    sampler=test_sampler,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

# =========================
# Utils: base model (hindari prefix 'module.')
# =========================
def _base_model(m):
    return m.module if isinstance(m, (torch.nn.DataParallel, torch.nn.parallel.DistributedDataParallel)) else m

# =========================
# Model / Optim / Scheduler (GLOBAL) / AMP
# =========================
num_labels = len(le.classes_)
hf_config = AutoConfig.from_pretrained(MODEL_ID, num_labels=num_labels)
hf_config.id2label = id2label
hf_config.label2id = label2id

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, config=hf_config)
model.to(device)

# --- Multi-GPU (DataParallel) ---
if torch.cuda.device_count() > 1:
    print(f"[INFO] Using DataParallel over {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

# AdamW fused (fallback bila tidak tersedia)
try:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
    log_report("[OPT] AdamW fused enabled")
except TypeError:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    log_report("[OPT] AdamW fused not available; fallback to standard AdamW")

scaler = GradScaler(enabled=AMP_ENABLED)

# Label smoothing loss (opsional)
loss_fn = None
if LABEL_SMOOTH > 0:
    import torch.nn as nn
    loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    log_report(f"[LOSS] Label smoothing = {LABEL_SMOOTH}")
else:
    log_report("[LOSS] Label smoothing = OFF")

# EMA
try:
    from torch_ema import ExponentialMovingAverage
    EMA_AVAILABLE = True
except Exception:
    EMA_AVAILABLE = False

ema = None
if USE_EMA and EMA_AVAILABLE:
    base_model = _base_model(model)
    ema = ExponentialMovingAverage(base_model.parameters(), decay=EMA_DECAY)
    log_report(f"[EMA] Enabled (decay={EMA_DECAY})")
elif USE_EMA and not EMA_AVAILABLE:
    log_report("[EMA] Requested but torch-ema not available → EMA OFF")
else:
    log_report("[EMA] OFF")

def flat_accuracy(logits_np, labels_np):
    preds = np.argmax(logits_np, axis=1).flatten()
    return float((preds == labels_np.flatten()).mean())

# ==== GLOBAL STEPS & SCHEDULER ====
_tmp_loader = DataLoader(
    train_data, sampler=RandomSampler(train_data), batch_size=BATCH_SIZE,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2
)
steps_per_epoch = len(_tmp_loader)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

if SCHEDULER.lower() == "cosine":
    from transformers import get_cosine_schedule_with_warmup
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
else:
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

log_report(f"Steps/epoch      : {steps_per_epoch}")
log_report(f"Total steps      : {total_steps}")
log_report(f"Warmup steps     : {warmup_steps}")
log_report("================================\n")

global_step = 0

# =========================
# Checkpoint helpers (DP-safe)
# =========================
def save_checkpoint(epoch, model, optimizer, scaler, best_val_loss, path=CKPT_PATH,
                    scheduler=None, global_step=0,
                    best_epoch=None, best_global_step=None,
                    best_val_f1=None, best_f1_epoch=None, best_f1_global_step=None,
                    ema_obj=None):
    m_save = _base_model(model)
    state = {
        "epoch": epoch,
        "model_state": m_save.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict() if AMP_ENABLED else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "global_step": global_step,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_global_step": best_global_step,
        "best_val_f1": best_val_f1,
        "best_f1_epoch": best_f1_epoch,
        "best_f1_global_step": best_f1_global_step,
        "model_id": MODEL_ID,
        "num_labels": num_labels,
        "seed": SEED,
        "amp": AMP_ENABLED,
        "ema_state": (ema_obj.state_dict() if ema_obj is not None else None),
    }
    torch.save(state, path)

def load_checkpoint(path=CKPT_PATH, device=device):
    ckpt = torch.load(path, map_location=device)
    cfg = AutoConfig.from_pretrained(ckpt["model_id"], num_labels=ckpt["num_labels"])
    m = AutoModelForSequenceClassification.from_config(cfg)
    m.load_state_dict(ckpt["model_state"])
    m.to(device)

    try:
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
    except TypeError:
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    opt.load_state_dict(ckpt["optimizer_state"])

    sc = GradScaler(enabled=AMP_ENABLED)
    if AMP_ENABLED and ckpt.get("scaler_state") is not None:
        try:
            sc.load_state_dict(ckpt["scaler_state"])
        except Exception:
            pass

    start_ep    = ckpt["epoch"]
    gstep       = ckpt.get("global_step", 0)
    sched_state = ckpt.get("scheduler_state", None)

    best_loss   = ckpt.get("best_val_loss", float("inf"))
    best_ep     = ckpt.get("best_epoch", None)
    best_gstep  = ckpt.get("best_global_step", None)

    best_f1     = ckpt.get("best_val_f1", None)
    best_f1_ep  = ckpt.get("best_f1_epoch", None)
    best_f1_gs  = ckpt.get("best_f1_global_step", None)

    ema_state = ckpt.get("ema_state", None)
    return (start_ep, m, opt, sc, best_loss, gstep, sched_state,
            best_ep, best_gstep, best_f1, best_f1_ep, best_f1_gs, ema_state)

# =========================
# Resume (only via RESUME env)
# =========================
best_val_loss = float("inf")
best_epoch = None
best_global_step = None

best_val_f1 = None
best_f1_epoch = None
best_f1_global_step = None

patience_count = 0
start_epoch = 0

RESUME = os.environ.get("RESUME", "0") == "1"
if RESUME:
    try:
        (start_epoch, model, optimizer, scaler,
         best_val_loss, global_step, sched_state,
         best_epoch, best_global_step,
         best_val_f1, best_f1_epoch, best_f1_global_step, ema_state) = load_checkpoint(CKPT_PATH, device)

        # re-wrap DP if needed
        if torch.cuda.device_count() > 1 and not isinstance(model, torch.nn.DataParallel):
            model = torch.nn.DataParallel(model)

        # rebuild ema object & load its state
        if USE_EMA and EMA_AVAILABLE:
            ema = ExponentialMovingAverage(_base_model(model).parameters(), decay=EMA_DECAY)
            if ema_state is not None:
                try:
                    ema.load_state_dict(ema_state)
                except Exception:
                    print("[RESUME] ema_state load failed, continue without EMA state")

        start_epoch = int(start_epoch) + 1
        if sched_state is not None:
            try:
                scheduler.load_state_dict(sched_state)
            except Exception as _:
                print("[RESUME] scheduler_state load failed, continue with fresh scheduler.")

        print(f"[RESUME] start_epoch={start_epoch}, "
              f"best_val_loss={best_val_loss:.6f}, best_epoch={best_epoch}, "
              f"best_val_f1={best_val_f1}, best_f1_epoch={best_f1_epoch}, "
              f"amp={AMP_ENABLED}, global_step={global_step}")
    except Exception as e:
        print(f"[RESUME] failed: {e}. Starting fresh.")
        best_val_loss = float("inf")
        start_epoch = 0
        global_step = 0
        best_val_f1 = None

# =========================
# Train loop + Early Stop + AMP + Grad Clip
# =========================
for epoch in range(start_epoch, EPOCHS):
    train_dataloader = build_train_loader()  # reshuffle tiap epoch
    model.train()
    tr_loss = 0.0
    nb_tr_steps = 0

    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=True,
        dynamic_ncols=True
    )

    for step, batch in enumerate(progress_bar):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=AMP_ENABLED):
            if loss_fn is None:
                outputs = model(
                    input_ids=b_input_ids,
                    attention_mask=b_input_mask,
                    labels=b_labels
                )
                loss = outputs.loss
                logits = outputs.logits
            else:
                outputs = model(
                    input_ids=b_input_ids,
                    attention_mask=b_input_mask
                )
                logits = outputs.logits
                loss = loss_fn(logits, b_labels)

            if loss.dim() > 0:
                loss = loss.mean()

        scaler.scale(loss).backward()

        # ---- GRADIENT CLIPPING (AMP-safe) ----
        if AMP_ENABLED:
            scaler.unscale_(optimizer)
        if MAX_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_NORM)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()      # scheduler GLOBAL di-step tiap batch
        global_step += 1

        # ---- EMA update setelah optimizer step ----
        if ema is not None:
            ema.update()

        tr_loss += loss.item()
        nb_tr_steps += 1

        if step % 10 == 0:
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        if (step + 1) % 200 == 0:
            avg_loss = tr_loss / max(1, nb_tr_steps)
            print(f"[heartbeat] epoch={epoch+1}, step={step+1}, avg_loss={avg_loss:.4f}", flush=True)

    avg_train_loss = tr_loss / max(1, nb_tr_steps)

    # === LOG train loss per epoch ===
    log_report(f"Epoch {epoch+1}/{EPOCHS}")
    log_report(f"Train loss: {avg_train_loss}")

    # ---- Validation ----
    model.eval()

    # swap ke EMA weights untuk evaluasi
    ema_ctx = False
    if ema is not None:
        ema.store()
        ema.copy_to(_base_model(model).parameters())
        ema_ctx = True

    val_loss_sum = 0.0
    val_acc_sum = 0.0
    nb_eval_steps = 0

    val_preds_epoch = []
    val_trues_epoch = []

    with torch.no_grad():
        for batch in validation_dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            with autocast(enabled=AMP_ENABLED):
                if loss_fn is None:
                    outputs = model(
                        input_ids=b_input_ids,
                        attention_mask=b_input_mask,
                        labels=b_labels
                    )
                    loss = outputs.loss
                    logits = outputs.logits
                else:
                    outputs = model(
                        input_ids=b_input_ids,
                        attention_mask=b_input_mask
                    )
                    logits = outputs.logits
                    loss = loss_fn(logits, b_labels)

                if loss.dim() > 0:
                    loss = loss.mean()

            val_loss_sum += loss.item()

            logits_np = logits.detach().cpu().numpy()
            labels_np = b_labels.detach().cpu().numpy()
            preds_np = np.argmax(logits_np, axis=1)

            val_preds_epoch.extend(preds_np.tolist())
            val_trues_epoch.extend(labels_np.tolist())

            val_acc_sum += float((preds_np.flatten() == labels_np.flatten()).mean())
            nb_eval_steps += 1

    avg_val_loss = val_loss_sum / max(1, nb_eval_steps)
    avg_val_acc  = val_acc_sum  / max(1, nb_eval_steps)
    val_f1_micro = f1_score(val_trues_epoch, val_preds_epoch, average="micro")

    # restore weights asli setelah evaluasi
    if ema_ctx:
        ema.restore()

    # === LOG ===
    log_report(f"Validation Loss: {avg_val_loss}")
    log_report(f"Validation Accuracy: {avg_val_acc}")
    log_report(f"Validation F1 (micro): {val_f1_micro}")
    log_report("")

    # ---- 1) EARLY STOP pakai LOSS ----
    improvement_loss = best_val_loss - avg_val_loss
    if improvement_loss > MIN_DELTA:
        best_val_loss = avg_val_loss
        patience_count = 0
        best_epoch = epoch
        best_global_step = global_step
    else:
        patience_count += 1
        print(f"[NO IMPROVE-LOSS] epoch {epoch}: val_loss={avg_val_loss:.6f} "
              f"(best={best_val_loss:.6f}) → patience {patience_count}/{PATIENCE}")
        if patience_count >= PATIENCE:
            print("[EARLY STOP] no significant improvement within patience. Stopping.")
            save_checkpoint(
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                best_val_loss=best_val_loss,
                path=CKPT_PATH,
                scheduler=scheduler,
                global_step=global_step,
                best_epoch=best_epoch,
                best_global_step=best_global_step,
                best_val_f1=best_val_f1,
                best_f1_epoch=best_f1_epoch,
                best_f1_global_step=best_f1_global_step,
                ema_obj=ema
            )
            break

    # ---- 2) SAVE BEST by F1 → SIMPAN ARTEFAK (config, tokenizer, state_dict) ----
    is_better_f1 = (best_val_f1 is None) or (val_f1_micro > best_val_f1 + 1e-12)
    if is_better_f1:
        best_val_f1 = val_f1_micro
        best_f1_epoch = epoch
        best_f1_global_step = global_step

        # pastikan yang disimpan = bobot EMA (sama dengan yang dievaluasi)
        if ema is not None:
            ema.store()
            ema.copy_to(_base_model(model).parameters())

        _bm = _base_model(model)
        # update mapping ke config
        _bm.config.id2label = id2label
        _bm.config.label2id = label2id

        # 1) save config & tokenizer ke dir versi
        _bm.config.architectures = ["BertForSequenceClassification"]
        os.makedirs(CONFIG_DIR, exist_ok=True)
        os.makedirs(TOKENIZER_DIR, exist_ok=True)
        _bm.config.save_pretrained(CONFIG_DIR)
        tokenizer.save_pretrained(TOKENIZER_DIR)

        # 2) save weights .pth (state_dict)
        torch.save(_bm.state_dict(), WEIGHTS_PATH)

        # 3) update pointer "latest" (hapus & copy ulang)
        for src, dst in [
            (CONFIG_DIR, CONFIG_DIR_LATEST),
            (TOKENIZER_DIR, TOKENIZER_DIR_LATEST),
        ]:
            if os.path.exists(dst):
                shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
        shutil.copyfile(WEIGHTS_PATH, WEIGHTS_PATH_LATEST)

        if ema is not None:
            ema.restore()

        print(f"[BEST-F1] epoch {epoch}: val_f1_micro={best_val_f1:.6f} "
              f"(saved config/tokenizer/weights VER={VERSION} + latest pointers)")

    # ---- 3) SAVE CHECKPOINT LAST ----
    save_checkpoint(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        best_val_loss=best_val_loss,
        path=CKPT_PATH,
        scheduler=scheduler,
        global_step=global_step,
        best_epoch=best_epoch,
        best_global_step=best_global_step,
        best_val_f1=best_val_f1,
        best_f1_epoch=best_f1_epoch,
        best_f1_global_step=best_f1_global_step,
        ema_obj=ema
    )

# =========================
# Helper load untuk evaluasi akhir (pakai artefak barusan disave)
# =========================
def load_for_eval(config_dir, tokenizer_dir, weights_path, device, num_labels):
    tokenizer_eval = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
    cfg_disk = AutoConfig.from_pretrained(config_dir)
    cfg_eval = AutoConfig.from_pretrained(config_dir, num_labels=num_labels or getattr(cfg_disk, "num_labels", None))
    model_eval = AutoModelForSequenceClassification.from_config(cfg_eval)
    sd = torch.load(weights_path, map_location=device)
    model_eval.load_state_dict(sd, strict=True)
    model_eval.to(device).eval()
    return model_eval, tokenizer_eval

# =========================
# Load BEST (by F1) untuk Validation/Test reports
# =========================
model, tokenizer = load_for_eval(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device, num_labels=len(le.classes_))
tags_vals = list(le.classes_)

# =========================
# Validation evaluation (BEST model only) + CSV
# =========================
val_predictions, val_true_labels = [], []
with torch.no_grad():
    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with autocast(enabled=AMP_ENABLED):
            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_input_mask
            )
            logits = outputs.logits

        logits_np = logits.detach().cpu().numpy()
        labels_np = b_labels.detach().cpu().numpy()
        preds = np.argmax(logits_np, axis=1)
        val_predictions.extend(preds.tolist())
        val_true_labels.extend(labels_np.tolist())

val_pred_tags = [tags_vals[p] for p in val_predictions]
val_true_tags = [tags_vals[l] for l in val_true_labels]

val_clf_rep = classification_report(val_true_tags, val_pred_tags, zero_division=0)
print(val_clf_rep)
evaluation_report.write("\n[Validation - BEST model (by F1)]\n")
evaluation_report.write(val_clf_rep + "\n")
evaluation_report.flush()

val_clf_dict = classification_report(val_true_tags, val_pred_tags, output_dict=True, zero_division=0)
val_clf_df = pd.DataFrame(val_clf_dict).transpose()
val_clf_csv_path = f"{config.EVALUATION_REPORT_PATH}/val_classification_report_{VERSION}.csv"
val_clf_df.to_csv(val_clf_csv_path, index=True)
print(f"Validation classification report CSV saved to: {val_clf_csv_path}")

# =========================
# Test evaluation (BEST model) + CSV
# =========================
predictions = []
true_labels = []
eval_loss = 0.0
eval_accuracy = 0.0
nb_eval_steps = 0

with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with autocast(enabled=AMP_ENABLED):
            if LABEL_SMOOTH > 0:
                outputs = model(
                    input_ids=b_input_ids,
                    attention_mask=b_input_mask
                )
                logits = outputs.logits
                loss = torch.nn.functional.cross_entropy(
                    logits, b_labels, label_smoothing=LABEL_SMOOTH
                )
            else:
                outputs = model(
                    input_ids=b_input_ids,
                    attention_mask=b_input_mask,
                    labels=b_labels
                )
                loss = outputs.loss
                logits = outputs.logits

            if loss.dim() > 0:
                loss = loss.mean()

        logits_np = logits.detach().cpu().numpy()
        labels_np = b_labels.detach().cpu().numpy()

        preds = np.argmax(logits_np, axis=1)
        predictions.extend(preds.tolist())
        true_labels.extend(labels_np.tolist())

        eval_loss += loss.item()
        eval_accuracy += float((preds == labels_np.flatten()).mean())
        nb_eval_steps += 1

avg_test_loss = eval_loss / max(1, nb_eval_steps)
avg_test_acc  = eval_accuracy / max(1, nb_eval_steps)

pred_tags  = [tags_vals[p] for p in predictions]
valid_tags = [tags_vals[l] for l in true_labels]
test_f1_micro = f1_score(valid_tags, pred_tags, average="micro")

print(f"Test loss: {avg_test_loss}")
print(f"Test Accuracy: {avg_test_acc}")
print(f"Test F1-Score (micro): {test_f1_micro}")

evaluation_report.write("\nFinal (Test)\n")
evaluation_report.write(f"Test loss: {avg_test_loss}\n")
evaluation_report.write(f"Test Accuracy: {avg_test_acc}\n")
evaluation_report.write(f"Test F1-Score (micro): {test_f1_micro}\n\n\n")
evaluation_report.flush()

clf_rep = classification_report(valid_tags, pred_tags, zero_division=0)
print(clf_rep)
evaluation_report.write("Evaluation Report (Test)\n\n")
evaluation_report.write(clf_rep + "\n")
evaluation_report.flush()

# =========================
# Confusion matrix (CSV)
# =========================
cm = confusion_matrix(valid_tags, pred_tags, labels=tags_vals)
cm_df = pd.DataFrame(cm, index=tags_vals, columns=tags_vals)
cm_path = f"{config.CONFUSION_MATRIX_PATH}/confusion_matrix_{VERSION}.csv"
cm_df.to_csv(cm_path, index=True)

evaluation_report.close()

print(f"\nSaved BEST artifacts (by F1):")
print(f" - CONFIG_DIR    : {CONFIG_DIR}")
print(f" - TOKENIZER_DIR : {TOKENIZER_DIR}")
print(f" - WEIGHTS_PATH  : {WEIGHTS_PATH}")
print(f"Also updated 'latest' pointers:")
print(f" - {CONFIG_DIR_LATEST}")
print(f" - {TOKENIZER_DIR_LATEST}")
print(f" - {WEIGHTS_PATH_LATEST}")
print(f"Saved checkpoint (last state) to: {CKPT_PATH}")
print(f"Saved report to: {report_path}")
print(f"Saved confusion matrix to: {cm_path}")
print(f"AMP enabled: {AMP_ENABLED}")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

===== Training Parameters =====
Model ID         : bert-base-multilingual-uncased
Device / AMP     : cuda / True
Seed             : 42
Max length       : 512
Batch size       : 64 (total)
Learning rate    : 2e-05
Weight decay     : 0.01
Warmup ratio     : 0.1
Epochs (max)     : 10
Patience         : 2
Min delta (loss) : 0.001
Scheduler        : cosine
Grad clip maxnorm: 1.0
Label smoothing  : 0.0
EMA              : ON (decay=0.999)
Artifacts VER    : 20250915171406
Total Subchannel: 84
['bisnis__agribisnis', 'bisnis__ekonomi', 'bisnis__ekonomi_digital', 'bisnis__industri', 'bisnis__infrastruktur', 'bisnis__karier_keuangan', 'bisnis__korporasi', 'bisnis__market', 'bisnis__properti', 'bisnis__tambang_energi', 'bisnis__transportasi', 'bisnis__umkm', 'bola__sports__basket', 'bola__sports__formula_1', 'bola__sports__liga_champions', 'bola__sports__liga_europa', 'bola__sports__liga_indonesia', 'bola__sports__liga_inggris', 'bola__sports__liga_internasional', 'bola__sports__liga_italia', 'bol

...

Epoch 6/10
Train loss: 0.10305326507311112
Validation Loss: 0.28558249124325813
Validation Accuracy: 0.9192296006944445
Validation F1 (micro): 0.9192139737991266

[NO IMPROVE-LOSS] epoch 5: val_loss=0.285582 (best=0.247080) → patience 2/2
[EARLY STOP] no significant improvement within patience. Stopping.


/tmp/ipykernel_36/3377031080.py:721: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):


💤 keepalive 2025-09-15 23:01:28                                  precision    recall  f1-score   support

              bisnis__agribisnis       0.86      0.92      0.89       390
                 bisnis__ekonomi       0.84      0.83      0.84       530
         bisnis__ekonomi_digital       0.83      0.74      0.79       231
                bisnis__industri       0.84      0.83      0.83       281
           bisnis__infrastruktur       0.90      0.86      0.88       356
         bisnis__karier_keuangan       0.86      0.80      0.83       237
               bisnis__korporasi       0.80      0.80      0.80       564
                  bisnis__market       0.94      0.93      0.94       479
                bisnis__properti       0.87      0.88      0.88       245
          bisnis__tambang_energi       0.91      0.92      0.91       581
            bisnis__transportasi       0.88      0.92      0.90       365
                    bisnis__umkm       0.76      0.84      0.80       176
      

/tmp/ipykernel_36/3377031080.py:764: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):


KeyboardInterrupt: 

In [5]:
# %% [code] Minimal inference loader (config + tokenizer + state_dict)
import os, math, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from torch.cuda.amp import autocast

from config import config  # <- pastikan sama seperti saat training

# ---------------------------
# Device & params
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = (device.type == "cuda")

MAX_LEN    = int(config.MAX_LEN)
BATCH_SIZE = int(config.BATCH_SIZE)
VERSION    = datetime.now().strftime("%d%m%Y%H%M%S")  # boleh diganti string manual kalau mau

# ---------------------------
# Paths (sesuaikan dengan environment)
# ---------------------------
CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_20250915171406"
TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_20250915171406"
WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_20250915171406.pth"

TEST_CSV      = f"{config.TEST_SET_PATH}/{config.TEST_SET_FILENAME}"

assert os.path.exists(CONFIG_DIR),    f"CONFIG_DIR tidak ditemukan: {CONFIG_DIR}"
assert os.path.exists(TOKENIZER_DIR), f"TOKENIZER_DIR tidak ditemukan: {TOKENIZER_DIR}"
assert os.path.exists(WEIGHTS_PATH),  f"WEIGHTS_PATH tidak ditemukan: {WEIGHTS_PATH}"
assert os.path.exists(TEST_CSV),      f"TEST_CSV tidak ditemukan: {TEST_CSV}"

# ---------------------------
# Load model + tokenizer dari artefak deployment
# ---------------------------
def load_model_from_artifacts(config_dir, tokenizer_dir, weights_path, device):
    cfg = AutoConfig.from_pretrained(config_dir)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    # Pastikan arsitektur task classification
    if getattr(cfg, "architectures", None) is None:
        cfg.architectures = ["BertForSequenceClassification"]

    model = AutoModelForSequenceClassification.from_config(cfg)
    state_dict = torch.load(weights_path, map_location=device)

    # strict=False biar aman kalau ada key non-kritis (misal dropout, bias)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print("[WARN] state_dict mismatch:",
              f"missing={missing}", f"unexpected={unexpected}")

    model.to(device).eval()
    return model, tokenizer, cfg

model, tokenizer, cfg = load_model_from_artifacts(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device)

# Ambil mapping label dari config (gantikan label_encoder.pkl)
id2label = {int(k): v for k, v in getattr(cfg, "id2label", {}).items()}
if not id2label:
    raise RuntimeError("id2label kosong di config. Pastikan mapping disimpan saat training.")
num_labels = len(id2label)

# (opsional) sanity check dim classifier sesuai jumlah label
try:
    w = model.classifier.weight
    assert w.shape[0] == num_labels, f"Mismatch num_labels: model={w.shape[0]} vs cfg={num_labels}"
except Exception:
    pass  # beberapa model punya head berbeda penamaan; aman lanjut kalau tidak tersedia

# ---------------------------
# Encode helper (chunked supaya hemat RAM)
# ---------------------------
def build_loader(texts, tokenizer, batch_size=BATCH_SIZE, max_len=MAX_LEN, desc="Tokenizing"):
    all_input_ids, all_attn = [], []
    chunk = 10000
    n = len(texts)
    for i in tqdm(range(0, n, chunk), total=math.ceil(n/chunk), desc=desc):
        batch = list(map(str, texts[i:i+chunk]))
        enc = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_attention_mask=True,
            return_tensors="np",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
    input_ids = torch.tensor(np.vstack(all_input_ids), dtype=torch.long)
    attn_mask = torch.tensor(np.vstack(all_attn), dtype=torch.long)
    ds = TensorDataset(input_ids, attn_mask)
    return DataLoader(ds, sampler=SequentialSampler(ds), batch_size=batch_size)

# ---------------------------
# Predict helper → kembalikan DataFrame
# ---------------------------
def predict_df(df, text_col="content", title_col="title"):
    assert text_col in df.columns, f"Kolom '{text_col}' tidak ada di DataFrame."
    titles = df[title_col].values if title_col in df.columns else [None]*len(df)

    loader = build_loader(df[text_col].values, tokenizer, max_len=MAX_LEN,
                          batch_size=BATCH_SIZE, desc="Tokenizing (infer)")

    logits_list = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch[0].to(device)
            mask = batch[1].to(device)
            with autocast(enabled=AMP_ENABLED):
                out = model(input_ids=ids, attention_mask=mask)
                logits = out.logits
            logits_list.append(logits.detach().cpu().numpy())

    logits = np.concatenate(logits_list, axis=0)

    # Softmax stabil
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    # Top-1
    top1_idx = np.argmax(probs, axis=1)
    top1_pr  = probs[np.arange(len(top1_idx)), top1_idx]

    # (opsional) Top-2
    top2_idx = np.argsort(-probs, axis=1)[:, 1]
    top2_pr  = probs[np.arange(len(top2_idx)), top2_idx]

    out = pd.DataFrame({
        text_col: df[text_col].values,
        "title": titles,
        "pred_subchannel": [id2label[i] for i in top1_idx],
        "prob_pred": top1_pr,
        "second_subchannel": [id2label[i] for i in top2_idx],
        "prob_second": top2_pr,
    })

    # Kalau punya ground-truth di df (opsional, hanya untuk analisis)
    if "subchannel" in df.columns:
        out.insert(2, "true_subchannel", df["subchannel"].values)

    return out

# ---------------------------
# Run on TEST_CSV & simpan CSV
# ---------------------------
test_df = pd.read_csv(TEST_CSV)
df_test_out  = predict_df(test_df, text_col="content", title_col="title")

pred_csv_path = f"{config.EVALUATION_REPORT_PATH}/test_predictions_{VERSION}.csv"
os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
df_test_out.to_csv(pred_csv_path, index=False)
print(f"Saved detailed test predictions to: {pred_csv_path}")

df_test_out.head()


Tokenizing (infer): 100%|██████████| 7/7 [00:54<00:00,  7.76s/it]
/tmp/ipykernel_36/1274025496.py:115: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):


KeyboardInterrupt: 

💤 keepalive 2025-09-15 23:21:28

In [6]:
# %% [code] Minimal inference loader (config + tokenizer + state_dict) + metrics
import os, math, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from torch.cuda.amp import autocast

from config import config  # <- pastikan sama dengan saat training

# ---------------------------
# Device & params
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = (device.type == "cuda")

MAX_LEN    = int(config.MAX_LEN)
BATCH_SIZE = int(config.BATCH_SIZE)
VERSION    = datetime.now().strftime("%d%m%Y%H%M%S")  # bisa ganti string manual kalau mau

# ---------------------------
# Paths (ubah sesuai artefak model yang dipakai)
# ---------------------------
CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_20250915171406"
TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_20250915171406"
WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_20250915171406.pth"

TEST_CSV      = f"{config.TEST_SET_PATH}/{config.TEST_SET_FILENAME}"
VAL_CSV       = f"{config.VALIDATION_SET_PATH}/{config.VALIDATION_SET_FILENAME}"

# ---------------------------
# Load model + tokenizer dari artefak deployment
# ---------------------------
def load_model_from_artifacts(config_dir, tokenizer_dir, weights_path, device):
    cfg = AutoConfig.from_pretrained(config_dir)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    # Pastikan arsitektur task classification
    if getattr(cfg, "architectures", None) is None:
        cfg.architectures = ["BertForSequenceClassification"]

    model = AutoModelForSequenceClassification.from_config(cfg)
    state_dict = torch.load(weights_path, map_location=device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print("[WARN] state_dict mismatch:",
              f"missing={missing}", f"unexpected={unexpected}")

    model.to(device).eval()
    return model, tokenizer, cfg

model, tokenizer, cfg = load_model_from_artifacts(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device)

# Mapping label dari config
id2label = {int(k): v for k, v in getattr(cfg, "id2label", {}).items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

# ---------------------------
# Encode helper
# ---------------------------
def build_loader(texts, tokenizer, batch_size=BATCH_SIZE, max_len=MAX_LEN, desc="Tokenizing"):
    all_input_ids, all_attn = [], []
    chunk = 10000
    n = len(texts)
    for i in tqdm(range(0, n, chunk), total=math.ceil(n/chunk), desc=desc):
        batch = list(map(str, texts[i:i+chunk]))
        enc = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_attention_mask=True,
            return_tensors="np",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
    input_ids = torch.tensor(np.vstack(all_input_ids), dtype=torch.long)
    attn_mask = torch.tensor(np.vstack(all_attn), dtype=torch.long)
    ds = TensorDataset(input_ids, attn_mask)
    return DataLoader(ds, sampler=SequentialSampler(ds), batch_size=batch_size)

# ---------------------------
# Predict helper → return DataFrame
# ---------------------------
def predict_df(df, split_name="test", text_col="content", title_col="title"):
    assert text_col in df.columns, f"Kolom '{text_col}' tidak ada di DataFrame."
    titles = df[title_col].values if title_col in df.columns else [None]*len(df)

    loader = build_loader(df[text_col].values, tokenizer, max_len=MAX_LEN,
                          batch_size=BATCH_SIZE, desc=f"Tokenizing ({split_name})")

    logits_list = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch[0].to(device)
            mask = batch[1].to(device)
            with autocast(enabled=AMP_ENABLED):
                out = model(input_ids=ids, attention_mask=mask)
                logits = out.logits
            logits_list.append(logits.detach().cpu().numpy())

    logits = np.concatenate(logits_list, axis=0)

    # Softmax stabil
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    top1_idx = np.argmax(probs, axis=1)
    top1_pr  = probs[np.arange(len(top1_idx)), top1_idx]

    top2_idx = np.argsort(-probs, axis=1)[:, 1]
    top2_pr  = probs[np.arange(len(top2_idx)), top2_idx]

    out = pd.DataFrame({
        text_col: df[text_col].values,
        "title": titles,
        "pred_subchannel": [id2label[i] for i in top1_idx],
        "prob_pred": top1_pr,
        "second_subchannel": [id2label[i] for i in top2_idx],
        "prob_second": top2_pr,
    })

    if "subchannel" in df.columns:
        out.insert(2, "true_subchannel", df["subchannel"].values)

        # Classification report
        y_true = df["subchannel"].values
        y_pred = [id2label[i] for i in top1_idx]
        report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
        report_df = pd.DataFrame(report).transpose()
        report_path = f"{config.EVALUATION_REPORT_PATH}/{split_name}_classification_report_{VERSION}.csv"
        report_df.to_csv(report_path)
        print(f"{split_name.capitalize()} classification report saved to: {report_path}")

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred, labels=list(id2label.values()))
        cm_df = pd.DataFrame(cm, index=list(id2label.values()), columns=list(id2label.values()))
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm_df, cmap="Blues", xticklabels=True, yticklabels=True)
        plt.title(f"Confusion Matrix ({split_name})")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        cm_path = f"{config.EVALUATION_REPORT_PATH}/{split_name}_confusion_matrix_{VERSION}.png"
        plt.tight_layout()
        plt.savefig(cm_path)
        plt.close()
        print(f"{split_name.capitalize()} confusion matrix saved to: {cm_path}")

    return out

# ---------------------------
# Run on TEST + VALIDATION
# ---------------------------
for split_name, csv_path in [("test", TEST_CSV), ("val", VAL_CSV)]:
    assert os.path.exists(csv_path), f"{split_name} CSV tidak ditemukan: {csv_path}"
    df = pd.read_csv(csv_path)

    df_out = predict_df(df, split_name=split_name, text_col="content", title_col="title")

    pred_csv_path = f"{config.EVALUATION_REPORT_PATH}/{split_name}_predictions_{VERSION}.csv"
    os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
    df_out.to_csv(pred_csv_path, index=False)
    print(f"Saved detailed {split_name} predictions to: {pred_csv_path}")
    display(df_out.head())


Tokenizing (test): 100%|██████████| 7/7 [00:53<00:00,  7.66s/it]
/tmp/ipykernel_36/29565503.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):


💤 keepalive 2025-09-15 23:31:28Test classification report saved to: /kaggle/working/reports/test_classification_report_15092025232316.csv
Test confusion matrix saved to: /kaggle/working/reports/test_confusion_matrix_15092025232316.png
Saved detailed test predictions to: /kaggle/working/reports/test_predictions_15092025232316.csv


,content,title,true_subchannel,pred_subchannel,prob_pred,second_subchannel,prob_second
0,mom moms harga emas jual beli emas moms keluar...,moms harga emas lagi naik! sebaiknya kita jual...,mom__dunia_ibu,mom__dunia_ibu,0.932617,mom__parenting,0.037323
1,news bentrok brimob vs tni al sorong tni al da...,Akhir Damai Bentrokan TNI AL vs Brimob di Sorong,news__kriminal,news__religi,0.946289,news__kriminal,0.040619
2,news bnpb erupsi gunung gunung kembar lewotobi...,Satu Orang Luka Imbas Erupsi Gunung Lewotobi L...,news__peristiwa,news__peristiwa,0.999023,news__kriminal,0.000407
3,news banten cleansing guru honorer guru guru h...,Kadindikbud Pastikan Tak Ada 'Cleansing' Guru ...,news__pendidikan,news__pendidikan,0.998047,news__kriminal,0.000230
4,bisnis hasil kawal stabilisasi harga pangan ke...,berhasil kawal stabilisasi harga pangan kement...,bisnis__agribisnis,bisnis__agribisnis,0.997070,bisnis__industri,0.000261


Tokenizing (val): 100%|██████████| 4/4 [00:26<00:00,  6.71s/it]
/tmp/ipykernel_36/29565503.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):


Val classification report saved to: /kaggle/working/reports/val_classification_report_15092025232316.csv
Val confusion matrix saved to: /kaggle/working/reports/val_confusion_matrix_15092025232316.png
Saved detailed val predictions to: /kaggle/working/reports/val_predictions_15092025232316.csv


,content,title,true_subchannel,pred_subchannel,prob_pred,second_subchannel,prob_second
0,mom alam alami plasenta previa hamil hai moms ...,pengalamanku alami plasenta previa di kehamila...,mom__kehamilan,mom__kehamilan,0.997070,mom__seks,0.000430
1,bola sports sena williams misi sena williams a...,serena williams misinya jadi yang terbaik sepa...,bola__sports__raket,bola__sports__raket,0.999023,bola__sports__olimpiade,0.000177
2,news etle ganjil genap jateng lebaran 2024 aru...,"Selama Arus Mudik dan Balik, Ada 1.416 Pelangg...",news__peristiwa,news__peristiwa,0.999023,news__kriminal,0.000247
3,bola sports main timnas u 23 piala asia klub l...,Pemain Timnas U-23 untuk Piala Asia: Satu Klub...,bola__sports__liga_indonesia,bola__sports__liga_indonesia,0.999023,bola__sports__liga_internasional,0.000133
4,bisnis ihsg sesi i kuat 0 36 persen saham tguk...,"IHSG Sesi I Menguat 0,36 Persen, Saham TGUK Me...",bisnis__market,bisnis__market,0.999023,bisnis__ekonomi,0.000234


💤 keepalive 2025-09-15 23:51:28

In [ ]:
# %% [code] Minimal inference loader (VALIDATION)
import os, math, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from torch.cuda.amp import autocast

from config import config  # sama dengan saat training

# ---------------------------
# Device & params
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = (device.type == "cuda")

MAX_LEN    = int(config.MAX_LEN)
BATCH_SIZE = int(config.BATCH_SIZE)
VERSION    = datetime.now().strftime("%d%m%Y%H%M%S")

# ---------------------------
# Paths
# ---------------------------
CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_20250915171406"
TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_20250915171406"
WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_20250915171406.pth"
VAL_CSV       = f"{config.VALIDATION_SET_PATH}/{config.VALIDATION_SET_FILENAME}"

assert os.path.exists(VAL_CSV), f"Validation CSV tidak ditemukan: {VAL_CSV}"

# ---------------------------
# Load model + tokenizer
# ---------------------------
def load_model_from_artifacts(config_dir, tokenizer_dir, weights_path, device):
    cfg = AutoConfig.from_pretrained(config_dir)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    if getattr(cfg, "architectures", None) is None:
        cfg.architectures = ["BertForSequenceClassification"]

    model = AutoModelForSequenceClassification.from_config(cfg)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device).eval()
    return model, tokenizer, cfg

model, tokenizer, cfg = load_model_from_artifacts(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device)

id2label = {int(k): v for k, v in getattr(cfg, "id2label", {}).items()}
if not id2label:
    raise RuntimeError("id2label kosong di config.json")

# ---------------------------
# Encode helper
# ---------------------------
def build_loader(texts, tokenizer, batch_size=BATCH_SIZE, max_len=MAX_LEN, desc="Tokenizing (val)"):
    all_input_ids, all_attn = [], []
    chunk = 10000
    n = len(texts)
    for i in tqdm(range(0, n, chunk), total=math.ceil(n/chunk), desc=desc):
        batch = list(map(str, texts[i:i+chunk]))
        enc = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_attention_mask=True,
            return_tensors="np",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
    input_ids = torch.tensor(np.vstack(all_input_ids), dtype=torch.long)
    attn_mask = torch.tensor(np.vstack(all_attn), dtype=torch.long)
    ds = TensorDataset(input_ids, attn_mask)
    return DataLoader(ds, sampler=SequentialSampler(ds), batch_size=batch_size)

# ---------------------------
# Predict helper
# ---------------------------
def predict_df(df, text_col="content", title_col="title"):
    assert text_col in df.columns
    titles = df[title_col].values if title_col in df.columns else [None]*len(df)

    loader = build_loader(df[text_col].values, tokenizer, max_len=MAX_LEN,
                          batch_size=BATCH_SIZE, desc="Tokenizing (val)")

    logits_list = []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch[0].to(device), batch[1].to(device)
            with autocast(enabled=AMP_ENABLED):
                out = model(input_ids=ids, attention_mask=mask)
                logits = out.logits
            logits_list.append(logits.detach().cpu().numpy())

    logits = np.concatenate(logits_list, axis=0)
    probs  = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs  = probs / probs.sum(axis=1, keepdims=True)

    top1_idx = np.argmax(probs, axis=1)
    top1_pr  = probs[np.arange(len(top1_idx)), top1_idx]
    top2_idx = np.argsort(-probs, axis=1)[:, 1]
    top2_pr  = probs[np.arange(len(top2_idx)), top2_idx]

    out = pd.DataFrame({
        text_col: df[text_col].values,
        "title": titles,
        "pred_subchannel": [id2label[i] for i in top1_idx],
        "prob_pred": top1_pr,
        "second_subchannel": [id2label[i] for i in top2_idx],
        "prob_second": top2_pr,
    })

    if "subchannel" in df.columns:
        out.insert(2, "true_subchannel", df["subchannel"].values)

    return out

# ---------------------------
# Run on VAL_CSV & save
# ---------------------------
val_df  = pd.read_csv(VAL_CSV)
df_val_out  = predict_df(val_df, text_col="content", title_col="title")

pred_csv_path = f"{config.EVALUATION_REPORT_PATH}/val_predictions_{VERSION}.csv"
os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
df_val_out.to_csv(pred_csv_path, index=False)
print(f"Saved detailed validation predictions to: {pred_csv_path}")

df_val_out.head()


💤 keepalive 2025-09-15 23:11:28

In [7]:
# %% [code] Per-row inference timing (10 samples, CPU only, artefak terbaru)
import os, time, math, torch
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

from config import config  # path dataset & output ambil dari sini

# ---------------------------
# Device & CPU threading
# ---------------------------
device = torch.device("cpu")     # paksa CPU
AMP_ENABLED = False              # AMP dimatikan di CPU
# (opsional) batasi thread biar stabil & reprodusibel
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
torch.set_num_threads(int(os.environ.get("TORCH_NUM_THREADS", "4")))

# ---------------------------
# Params
# ---------------------------
MAX_LEN    = int(config.MAX_LEN)
BATCH_SIZE = int(config.BATCH_SIZE)
VERSION    = datetime.now().strftime("%d%m%Y%H%M%S")

# ---------------------------
# Artefak model (latest pointers)
# ---------------------------
CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_20250915171406"
TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_20250915171406"
WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_20250915171406.pth"

# Data sumber
TEST_CSV = f"{config.TEST_SET_PATH}/{config.TEST_SET_FILENAME}"
assert os.path.exists(TEST_CSV), f"Test CSV tidak ditemukan: {TEST_CSV}"
assert os.path.isdir(CONFIG_DIR),    f"Config dir tidak ditemukan: {CONFIG_DIR}"
assert os.path.isdir(TOKENIZER_DIR), f"Tokenizer dir tidak ditemukan: {TOKENIZER_DIR}"
assert os.path.exists(WEIGHTS_PATH), f"Weights file tidak ditemukan: {WEIGHTS_PATH}"

# ---------------------------
# Load model + tokenizer dari artefak
# ---------------------------
def load_model_from_artifacts(config_dir, tokenizer_dir, weights_path, device):
    cfg = AutoConfig.from_pretrained(config_dir)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    # pastikan tipe arsitektur task classification
    if getattr(cfg, "architectures", None) is None:
        cfg.architectures = ["BertForSequenceClassification"]

    model = AutoModelForSequenceClassification.from_config(cfg)
    state_dict = torch.load(weights_path, map_location=device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print("[WARN] state_dict mismatch:",
              f"missing={missing}", f"unexpected={unexpected}")

    model.to(device).eval()
    return model, tokenizer, cfg

model, tokenizer, cfg = load_model_from_artifacts(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device)

# Mapping label dari config (ganti label_encoder.pkl)
id2label = {int(k): v for k, v in getattr(cfg, "id2label", {}).items()}
if not id2label:
    raise RuntimeError("id2label kosong di config. Pastikan mapping disimpan saat training.")

# sanity check head dimensi
num_labels_cfg = getattr(cfg, "num_labels", None)
head_out = model.classifier.weight.shape[0]
if num_labels_cfg is not None and head_out != num_labels_cfg:
    print(f"[WARN] head_out({head_out}) != cfg.num_labels({num_labels_cfg})")
if head_out != len(id2label):
    print(f"[WARN] head_out({head_out}) != len(id2label)({len(id2label)})")

# ---------------------------
# Ambil 10 sampel
# ---------------------------
full_df = pd.read_csv(TEST_CSV)
assert "content" in full_df.columns, "Kolom 'content' tidak ada di testset."
sample_df = full_df.sample(n=10, random_state=42).reset_index(drop=True)

# ---------------------------
# Helper encode 1 teks (per-row)
# ---------------------------
def encode_one(text: str, tokenizer, max_len: int):
    return tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

softmax = torch.nn.Softmax(dim=1)

# ---------------------------
# Per-row timing (CPU)
# ---------------------------
records = []
for i, row in sample_df.iterrows():
    text = str(row["content"])

    # timing tokenisasi
    t0 = time.perf_counter()
    enc = encode_one(text, tokenizer, MAX_LEN)
    waktu_tokenisasi_ms = (time.perf_counter() - t0) * 1000.0

    input_ids = enc["input_ids"].to(device)
    attn_mask = enc["attention_mask"].to(device)

    # timing forward (CPU)
    t1 = time.perf_counter()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
    waktu_inferensi_ms = (time.perf_counter() - t1) * 1000.0

    # prediksi top-1
    pred_idx  = int(torch.argmax(logits, dim=1).item())
    prob_pred = float(softmax(logits)[0, pred_idx].item())
    pred_label = id2label[pred_idx]

    total_waktu_ms = waktu_tokenisasi_ms + waktu_inferensi_ms

    records.append({
        "row_id": int(i),
        "content": text,  # full content
        "pred_subchannel": pred_label,
        "prob_pred": prob_pred,
        "waktu_tokenisasi_ms": waktu_tokenisasi_ms,
        "waktu_inferensi_ms": waktu_inferensi_ms,
        "total_waktu_ms": total_waktu_ms,
    })

per_row_df = pd.DataFrame(records)

# ---------------------------
# Ringkasan & Save
# ---------------------------
summary = per_row_df[["waktu_tokenisasi_ms","waktu_inferensi_ms","total_waktu_ms"]] \
    .agg(["mean","median","min","max"]).T
print("=== Ringkasan waktu (ms) [CPU] ===")
print(summary)

out_path = f"{config.EVALUATION_REPORT_PATH}/per_row_inference_10samples_cpu_{VERSION}.csv"
os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
per_row_df.to_csv(out_path, index=False)
print("Saved per-row timings (CPU) to:", out_path)

per_row_df.head()


=== Ringkasan waktu (ms) [CPU] ===
                           mean      median         min         max
waktu_tokenisasi_ms    2.452435    2.102758    1.549676    4.770084
waktu_inferensi_ms   478.795049  465.790951  453.885296  549.166990
total_waktu_ms       481.247484  469.138696  455.922844  550.828252
Saved per-row timings (CPU) to: /kaggle/working/reports/per_row_inference_10samples_cpu_15092025235722.csv


,row_id,content,pred_subchannel,prob_pred,waktu_tokenisasi_ms,waktu_inferensi_ms,total_waktu_ms
0,0,news kabar daerah news polisi polisi tembak ma...,news__kriminal,0.998064,1.549676,522.073382,523.623058
1,1,bisnis harga minyak mentah proyeksi sentuh usd...,bisnis__tambang_energi,0.998249,2.908825,461.101213,464.010038
2,2,news dpr golkar news pemilupedia 2024 golkar i...,news__politik,0.999434,1.661262,549.166990,550.828252
3,3,otomotif honda hr v hybrid jual rp 449 juta ha...,otomotif__motor,0.679973,2.167967,478.640533,480.808500
4,4,otomotif insentif pandemi jalan jual mobil mal...,otomotif__mobil,0.999222,3.426540,472.140902,475.567442


In [ ]:
# # %% [code] Inference-only for saved best model (top-1 & top-2 probs) [versi config+weights]
# import os
# import math
# import joblib
# import torch
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# from datetime import datetime

# from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
# from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
# from torch.cuda.amp import autocast

# from config import config  # pastikan sama dengan saat training

# # ---------------------------
# # Device & AMP
# # ---------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# AMP_ENABLED = (device.type == "cuda")

# # ---------------------------
# # Paths (pakai artifact versi training)
# # ---------------------------
# # VERSION       = datetime.now().strftime("%d%m%Y%H%M%S")  # ganti sesuai versi hasil training
# CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_latest"
# TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_latest"
# WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_latest.pth"
# LE_PATH       = f"{config.MODEL_PATH}/label_encoder.pkl"

# TEST_CSV      = f"{config.TEST_SET_PATH}/{config.TEST_SET_FILENAME}"
# MAX_LEN       = int(config.MAX_LEN)
# BATCH_SIZE    = int(config.BATCH_SIZE)

# assert os.path.exists(TEST_CSV), f"Test CSV tidak ditemukan: {TEST_CSV}"
# assert os.path.exists(LE_PATH), "label_encoder.pkl tidak ditemukan."
# assert os.path.exists(CONFIG_DIR), f"Config dir {CONFIG_DIR} tidak ada."
# assert os.path.exists(TOKENIZER_DIR), f"Tokenizer dir {TOKENIZER_DIR} tidak ada."
# assert os.path.exists(WEIGHTS_PATH), f"Weights {WEIGHTS_PATH} tidak ada."

# # ---------------------------
# # Load label encoder
# # ---------------------------
# le = joblib.load(LE_PATH)
# idx2label = {i: cls for i, cls in enumerate(le.classes_)}

# # ---------------------------
# # Helper: encode texts (chunked)
# # ---------------------------
# def build_test_loader(texts, batch_size=BATCH_SIZE, max_len=MAX_LEN, tokenizer=None):
#     all_input_ids, all_attn = [], []
#     chunk_size = 10000
#     n = len(texts)
#     for i in tqdm(range(0, n, chunk_size), desc="Tokenizing (test)", total=math.ceil(n/chunk_size)):
#         batch = list(map(str, texts[i:i+chunk_size]))
#         enc = tokenizer(
#             batch,
#             padding="max_length",
#             truncation=True,
#             max_length=max_len,
#             return_attention_mask=True,
#             return_tensors="np",
#         )
#         all_input_ids.append(enc["input_ids"])
#         all_attn.append(enc["attention_mask"])
#     input_ids  = torch.tensor(np.vstack(all_input_ids), dtype=torch.long)
#     attn_mask  = torch.tensor(np.vstack(all_attn), dtype=torch.long)
#     dataset    = TensorDataset(input_ids, attn_mask)
#     sampler    = SequentialSampler(dataset)
#     loader     = DataLoader(dataset, sampler=sampler, batch_size=batch_size)
#     return loader

# # ---------------------------
# # Loader (config + tokenizer + weights)
# # ---------------------------
# def load_model_for_inference(config_dir, tokenizer_dir, weights_path, device, num_labels=None):
#     tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
#     cfg = AutoConfig.from_pretrained(config_dir, num_labels=num_labels)
#     model = AutoModelForSequenceClassification.from_config(cfg)
#     state_dict = torch.load(weights_path, map_location=device)
#     model.load_state_dict(state_dict, strict=True)
#     model.to(device).eval()
#     return model, tokenizer

# # ---------------------------
# # Load model & tokenizer
# # ---------------------------
# expected_num_labels = len(le.classes_)
# model, tokenizer = load_model_for_inference(CONFIG_DIR, TOKENIZER_DIR, WEIGHTS_PATH, device, num_labels=expected_num_labels)

# w = model.classifier.weight
# print("classifier shape:", tuple(w.shape))
# assert w.shape[0] == expected_num_labels, "SALAH MODEL: num_labels tidak cocok."

# # ---------------------------
# # Load test data
# # ---------------------------
# testset = pd.read_csv(TEST_CSV)
# assert "content" in testset.columns, "Kolom 'content' tidak ada di testset."
# titles = testset["title"].values if "title" in testset.columns else [None]*len(testset)

# test_loader = build_test_loader(testset["content"].values, tokenizer=tokenizer, max_len=MAX_LEN, batch_size=BATCH_SIZE)

# # ---------------------------
# # Predict (top-1 & top-2 probabilities)
# # ---------------------------
# model.eval()
# preds_all, test_logits_list = [], []

# with torch.no_grad():
#     for batch in test_loader:
#         b_input_ids  = batch[0].to(device)
#         b_input_mask = batch[1].to(device)

#         with autocast(enabled=AMP_ENABLED):
#             outputs = model(input_ids=b_input_ids, attention_mask=b_input_mask)
#             logits = outputs.logits

#         test_logits_list.append(logits.detach().cpu().numpy())
#         preds_all.extend(torch.argmax(logits, dim=1).cpu().tolist())

# logits = np.concatenate(test_logits_list, axis=0)
# probs  = np.exp(logits - logits.max(axis=1, keepdims=True))
# probs  = probs / probs.sum(axis=1, keepdims=True)

# top1_idx = np.argmax(probs, axis=1)
# top1_pr  = probs[np.arange(len(top1_idx)), top1_idx]

# top2_idx = np.argsort(-probs, axis=1)[:, 1]
# top2_pr  = probs[np.arange(len(top2_idx)), top2_idx]

# # ---------------------------
# # Build output DataFrame
# # ---------------------------
# df_out = pd.DataFrame({
#     "content": testset["content"].values,
#     "title": titles,
#     "pred_subchannel": [idx2label[i] for i in top1_idx],
#     "prob_pred": top1_pr,
#     "second_subchannel": [idx2label[i] for i in top2_idx],
#     "prob_second": top2_pr,
# })

# if "subchannel" in testset.columns:
#     trues_all = le.transform(testset["subchannel"].values)
#     df_out.insert(2, "true_subchannel", [idx2label[i] for i in trues_all])

# # ---------------------------
# # Save & preview
# # ---------------------------
# pred_csv_path = f"{config.EVALUATION_REPORT_PATH}/test_predictions_{VERSION}.csv"
# os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
# df_out.to_csv(pred_csv_path, index=False)
# print(f"Saved detailed test predictions to: {pred_csv_path}")

# df_out.head()


In [ ]:
# # %% [code] Inference-only: VALIDATION predictions (top-1 & top-2) + CSV

# import os
# import math
# import joblib
# import torch
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
# from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
# from torch.cuda.amp import autocast

# from config import config  # harus sama dengan saat training

# # ---------------------------
# # Device & AMP
# # ---------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# AMP_ENABLED = (device.type == "cuda")

# # ---------------------------
# # Artefak model (IKUTI pola test sebelumnya)
# # ---------------------------
# # Gunakan artefak 'latest' agar stabil buat backfill/serving
# CONFIG_DIR    = f"{config.MODEL_PATH}/config_subchannel_latest"
# TOKENIZER_DIR = f"{config.MODEL_PATH}/tokenizer_subchannel_latest"
# WEIGHTS_PATH  = f"{config.MODEL_PATH}/model_subchannel_latest.pth"
# LE_PATH       = f"{config.MODEL_PATH}/label_encoder.pkl"

# # Data validation
# VAL_CSV   = f"{config.VALIDATION_SET_PATH}/{config.VALIDATION_SET_FILENAME}"
# MAX_LEN   = int(config.MAX_LEN)
# BATCH_SIZE= int(config.BATCH_SIZE)

# # Nama versi output (biar mudah dibaca manual)
# VERSION = "val_infer"
# # Kalau mau otomatis timestamp:
# # from datetime import datetime
# # VERSION = datetime.now().strftime("%d%m%Y%H%M%S")

# assert os.path.exists(VAL_CSV), f"Validation CSV tidak ditemukan: {VAL_CSV}"
# assert os.path.exists(LE_PATH), "label_encoder.pkl tidak ditemukan. Pastikan disimpan saat training."
# assert os.path.isdir(CONFIG_DIR), f"Folder config tidak ditemukan: {CONFIG_DIR}"
# assert os.path.isdir(TOKENIZER_DIR), f"Folder tokenizer tidak ditemukan: {TOKENIZER_DIR}"
# assert os.path.exists(WEIGHTS_PATH), f"File weights tidak ditemukan: {WEIGHTS_PATH}"

# # ---------------------------
# # Load label encoder & mapping
# # ---------------------------
# le = joblib.load(LE_PATH)
# tags_vals = le.classes_
# idx2label = {i: cls for i, cls in enumerate(tags_vals)}

# # ---------------------------
# # Load validation data
# # ---------------------------
# valset = pd.read_csv(VAL_CSV)
# assert "content" in valset.columns, "Kolom 'content' tidak ada di valset."
# titles = valset["title"].values if "title" in valset.columns else [None]*len(valset)

# # ---------------------------
# # Helper: encode texts (chunked, aman RAM)
# # ---------------------------
# def build_loader(texts, batch_size=BATCH_SIZE, max_len=MAX_LEN, tokenizer=None, desc="Tokenizing (val)"):
#     all_input_ids, all_attn = [], []
#     chunk_size = 10000
#     n = len(texts)
#     for i in tqdm(range(0, n, chunk_size), desc=desc, total=math.ceil(n/chunk_size)):
#         batch = list(map(str, texts[i:i+chunk_size]))
#         enc = tokenizer(
#             batch,
#             padding="max_length",
#             truncation=True,
#             max_length=max_len,
#             return_attention_mask=True,
#             return_tensors="np",
#         )
#         all_input_ids.append(enc["input_ids"])
#         all_attn.append(enc["attention_mask"])
#     input_ids  = torch.tensor(np.vstack(all_input_ids), dtype=torch.long)
#     attn_mask  = torch.tensor(np.vstack(all_attn), dtype=torch.long)
#     dataset    = TensorDataset(input_ids, attn_mask)
#     sampler    = SequentialSampler(dataset)
#     return DataLoader(dataset, sampler=sampler, batch_size=batch_size)

# # ---------------------------
# # Load model & tokenizer dari artefak
# # ---------------------------
# def load_model_from_artifacts(device):
#     # 1) config & tokenizer dari direktori
#     cfg = AutoConfig.from_pretrained(CONFIG_DIR)
#     tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)

#     # Pastikan jumlah label sesuai dengan label encoder
#     num_labels = len(le.classes_)
#     if getattr(cfg, "num_labels", None) != num_labels:
#         cfg.num_labels = num_labels

#     # 2) instansiasi model dari config (BUKAN from_pretrained)
#     model = AutoModelForSequenceClassification.from_config(cfg)

#     # 3) load bobot fine-tuned
#     state_dict = torch.load(WEIGHTS_PATH, map_location=device)
#     model.load_state_dict(state_dict, strict=True)

#     model.to(device).eval()
#     return model, tokenizer

# model, tokenizer = load_model_from_artifacts(device)

# # Sanity-check: head shape = (num_classes, hidden_size)
# expected_num_labels = len(le.classes_)
# w = model.classifier.weight
# print("classifier shape:", tuple(w.shape))
# assert w.shape[0] == expected_num_labels, "SALAH MODEL: weight head tidak cocok jumlah kelas."

# # ---------------------------
# # Build validation dataloader
# # ---------------------------
# validation_dataloader = build_loader(
#     valset["content"].values,
#     tokenizer=tokenizer,
#     max_len=MAX_LEN,
#     batch_size=BATCH_SIZE
# )

# # ---------------------------
# # Predict (top-1/top-2 probs)
# # ---------------------------
# model.eval()
# val_logits_list = []

# with torch.no_grad():
#     for batch in validation_dataloader:
#         ids  = batch[0].to(device)
#         mask = batch[1].to(device)

#         with autocast(enabled=AMP_ENABLED):
#             out = model(input_ids=ids, attention_mask=mask)
#             logits = out.logits

#         val_logits_list.append(logits.detach().cpu().numpy())

# logits = np.concatenate(val_logits_list, axis=0)

# # softmax stabil
# probs  = np.exp(logits - logits.max(axis=1, keepdims=True))
# probs  = probs / probs.sum(axis=1, keepdims=True)

# top1_idx = np.argmax(probs, axis=1)
# top1_pr  = probs[np.arange(len(top1_idx)), top1_idx]

# # top-2
# top2_idx = np.argsort(-probs, axis=1)[:, 1]
# top2_pr  = probs[np.arange(len(top2_idx)), top2_idx]

# # ---------------------------
# # Build DataFrame output
# # ---------------------------
# df_val_pred = pd.DataFrame({
#     "content": valset["content"].values,
#     "title": titles,
#     "pred_subchannel": [idx2label[i] for i in top1_idx],
#     "prob_pred": top1_pr,
#     "second_subchannel": [idx2label[i] for i in top2_idx],
#     "prob_second": top2_pr,
# })

# # kalau ground truth tersedia
# if "subchannel" in valset.columns:
#     true_idx = le.transform(valset["subchannel"].values)
#     df_val_pred.insert(2, "true_subchannel", [idx2label[i] for i in true_idx])

#     # (Optional) classification report ringkas di console
#     from sklearn.metrics import classification_report
#     y_true = [idx2label[i] for i in true_idx]
#     y_pred = [idx2label[i] for i in top1_idx]
#     print(classification_report(y_true, y_pred, zero_division=0))

# # ---------------------------
# # Save CSV
# # ---------------------------
# os.makedirs(config.EVALUATION_REPORT_PATH, exist_ok=True)
# VAL_PRED_PATH = f"{config.EVALUATION_REPORT_PATH}/val_predictions_{VERSION}.csv"
# df_val_pred.to_csv(VAL_PRED_PATH, index=False)
# print("Saved:", VAL_PRED_PATH)

# # preview
# df_val_pred.head()
